In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))
from paths import RAW, PROCESSED, RESULTS, savefig

Onshore Wind Eligible Areas

In [ ]:
import geopandas as gpd
import numpy as np
import rasterio
from atlite.gis import ExclusionContainer, shape_availability
from rasterio.plot import show

In [ ]:
# Regions produced by 01_regions_centroids.ipynb
gadm = gpd.read_file(PROCESSED / "gadm_aut_level1.geojson")

landcover_path = (
    RAW
    / "copernicus-glc"
    / "PROBAV_LC100_global_v3.0.1_2019-nrt_Discrete-Classification-map_EPSG-4326-AT.tif"
)

suitable_land_codes = [20, 30, 40, 60]

wdpa_path = RAW / "wdpa" / "WDPA_Oct2022_Public_shp-AUT.tif"

elevation_path = RAW / "gebco" / "GEBCO_2014_2D-AT.nc"

In [ ]:
wind_excluder = ExclusionContainer(crs=3035, res=100)

# protected areas
wind_excluder.add_raster(
    wdpa_path,
    nodata=255
)

# suitable land cover
wind_excluder.add_raster(
    landcover_path,
    codes=suitable_land_codes,
    invert=True,
    nodata=255
)

####### Airports

# Read airport data
airports_path = RAW / "ne_10m_airports.gpkg"
airports = gpd.read_file(airports_path)

# Project to metric CRS
airports_m = airports.to_crs("EPSG:3035")

# Austria regions in metric CRS
regions_m = gadm.to_crs("EPSG:3035")

# Keep only airports inside Austria
airports_aut = gpd.sjoin(
    airports_m,
    regions_m[["geometry"]],
    how="inner",
    predicate="within"
)

# airports buffer
wind_excluder.add_geometry(
    airports_aut.geometry,
    buffer=10_000
)

# Road buffer
roads_path = RAW / "ne_10m_roads.gpkg"
roads = gpd.read_file(roads_path)

roads_m = roads.to_crs("EPSG:3035")
regions_m = gadm.to_crs("EPSG:3035").set_index("NAME_1")

roads_aut = gpd.sjoin(
    roads_m,
    regions_m[["geometry"]],
    how="inner",
    predicate="intersects"
)

wind_excluder.add_geometry(
    roads_aut.geometry,
    buffer=300
)

# build up
wind_excluder.add_raster(
    landcover_path,
    codes=[50],
    buffer=1000,
    nodata=255
)

# elevation
wind_excluder.add_raster(
    elevation_path,
    codes=lambda x: x > 2000,
    crs=4326 # Manually specify CRS as EPSG:4326 for GEBCO data
)

In [ ]:
wind_availability, wind_transform = shape_availability(
    regions_m.geometry,
    wind_excluder
)

In [ ]:
# Export the eligibility mask for reuse by downstream notebooks (e.g. capacity factor / model building)
wind_mask_path = PROCESSED / "wind_availability_AUT.tif"
wind_data = wind_availability.astype("uint8")

with rasterio.open(
    wind_mask_path,
    "w",
    driver="GTiff",
    height=wind_data.shape[0],
    width=wind_data.shape[1],
    count=1,
    dtype=wind_data.dtype,
    crs="EPSG:3035",
    transform=wind_transform,
) as dst:
    dst.write(wind_data, 1)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 8))

show(
    wind_availability,
    transform=wind_transform,
    ax=ax
)

regions_m.boundary.plot(ax=ax, color="black", linewidth=0.8)

ax.set_title("Final Onshore Wind Eligible Areas in Austria")

savefig(fig, "land_eligibility", "wind_eligible_areas.png")
plt.show()